# Preprocesado de los datos

In [3]:
import json 
import re

## Carga de datos

In [42]:
with open("data/val_split.json", "r") as file:
    dataset = json.load(file)

In [43]:
print(dataset.keys())

dict_keys(['39166107', '40665707', '40042065', '28716445', '37849234', '39392298', '34205336', '33963227', '38480804', '37802772', '37403161', '33963281', '38151560', '37676767', '25869281', '33777957', '38850348', '28556833', '33195226', '30309367', '39019369', '37422636', '37451747', '39998261', '36253847', '37212075', '33327540', '39579960', '34480631', '36804535', '31662903', '31530799', '34290352', '35674870', '32438623', '37693595', '34560239', '36731694', '38613087', '37071196'])


## Tokenizado a nivel de palabra y símbolos

### Funciones

In [4]:
def tokenize_text_with_positions(text):
    # Split text into tokens, preserving punctuation (except for hyphens and underscores)
    tokens = []
    token_spans = []  # list of (start_char_index, end_char_index) for each token
    pattern = re.compile(
        r"</?[^>\s]+(?:\s+[^>]*)?>|\w+(?:[-_]\w+)*|[.,!?;:\'\"()\[\]{}<>]|[\s]+|\S"
    )
    for match in pattern.finditer(text):
        token = match.group()
        if token.isspace():
            continue  # Skip whitespace tokens
        start_pos = match.start()
        if re.match(r"\w+-\w+", token) or re.match(r"\w+_\w+", token) or re.match(r"</?[^>\s]+(?:\s+[^>]*)?>", token):
            # Keep hyphenated or underscored words intact
            tokens.append(token)
            token_spans.append((start_pos, match.end()))
        else:
            # Split contractions (e.g., "don't" -> "don", "'", "t")
            contraction_match = re.match(r"(\w+)(')(\w+)", token)
            if contraction_match:
                groups = contraction_match.groups()
                for group in groups:
                    end_pos = start_pos + len(group)
                    tokens.append(group)
                    token_spans.append((start_pos, end_pos))
                    start_pos = end_pos
            else:
                tokens.append(token)
                token_spans.append((start_pos, match.end()))
    return tokens, token_spans

In [5]:
def process_annotations(data):
    output_data = []

    for doc_id, doc_data in data.items():
        fields = ["title", "abstract"]

        for field in fields:
            text = doc_data["metadata"].get(field, "")
            tokens, token_spans = tokenize_text_with_positions(text)

            ner = []

            field_entities = []
            for entity in doc_data.get("entities", []):
                mention_location = entity.get("location", "")
                if mention_location == field:
                    field_entities.append(entity)

            for entity in field_entities:
                entity_start_char = entity["start_idx"]
                entity_end_char = entity["end_idx"] + 1
                entity_label = entity["label"]

                entity_start_token_index = None
                entity_end_token_index = None

                for i, (token_start_char, token_end_char) in enumerate(token_spans):
                    if token_end_char <= entity_start_char:
                        continue
                    if token_start_char >= entity_end_char:
                        break

                    if entity_start_token_index is None:
                        entity_start_token_index = i
                    entity_end_token_index = i

                if entity_start_token_index is not None and entity_end_token_index is not None:
                    ner.append([
                        entity_start_token_index,
                        entity_end_token_index,
                        entity_label.lower()
                    ])
                else:
                    print(f"Warning: Could not find tokens for entity in doc {doc_id}, field {field}")

            ner.sort(key=lambda x: x[0])

            output_doc = {
                "pmid": doc_id,
                "location": field,
                "tokenized_text": tokens,
                "ner": ner,
                "token_indexes": token_spans
            }

            output_data.append(output_doc)

    return output_data

In [6]:
def spans_a_io(example):  ### NO USAR
    tokens = example["tokenized_text"]
    ner = example["ner"]

    labels = ["O"] * len(tokens)

    for start, end, label in ner:

        # end es inclusivo
        for idx in range(start, end + 1):
            labels[idx] = label

    return labels

def spans_a_bio(tokens, spans):
    """
    tokens: lista de tokens
    spans: lista de tuplas/listas (start_token, end_token, label)
           end_token inclusivo
    """
    bio = ["O"] * len(tokens)

    for start, end, label in spans:
        bio[start] = f"B-{label}"

        for i in range(start + 1, end + 1):
            bio[i] = f"I-{label}"

    return bio

def bionizar_datos(data):
    bio_dataset = []
    for i, entry in enumerate(data):
        sample = {
            "pmid": entry["pmid"],
            "location":entry["location"]
        }
        sample["tokens"] = entry["tokenized_text"]
        
        lista = spans_a_bio(entry["tokenized_text"], entry["ner"])  
        sample["bio"] = lista
        sample["indexes"] = entry["token_indexes"]
        bio_dataset.append(sample)
    return bio_dataset
    

In [21]:
def chunking(entry, size = 256, overlap = 20):
    chunks = []
    tokens = entry["tokens"]
    token_spans = entry["indexes"]
    bio = entry["bio"]
    step = size - overlap
    for i in range(0, len(tokens), step):
        token_start = i
        token_end = i+ size
        chunk = tokens[token_start:token_end]
        chunk_spans = token_spans[token_start:token_end]
        chunk_bio = bio[token_start:token_end]
        chunk_start = token_spans[i][0]
        chunk_end = token_spans[i+len(chunk)-1][1]
        chunks.append({
            "pmid": entry["pmid"],
            "location": entry["location"],
            "chunk_start_char": chunk_start,
            "chunk_start_tok": token_start,
            "chunk_end_char":chunk_end,
            "chunk_end_tok":token_start + len(chunk),
            "chunk_tokens": chunk,
            "chunk_labels":chunk_bio,
            "chunk_spans":chunk_spans
        })
        if token_end >= len(tokens):
            break
    return chunks
def chunking_gliner(entry, size=384, overlap=20):
    chunks = []

    tokens = entry["tokenized_text"]
    ner = entry["ner"]

    step = size - overlap

    for i in range(0, len(tokens), step):
        token_start = i
        token_end = min(i + size, len(tokens))

        chunk_tokens = tokens[token_start:token_end]

        chunk_ner = []

        for ent_start, ent_end, label in ner:
            ent_start = int(ent_start)
            ent_end = int(ent_end)

            # Mantener solo entidades completamente dentro del chunk
            if ent_start >= token_start and ent_end < token_end:
                new_start = ent_start - token_start
                new_end = ent_end - token_start

                chunk_ner.append([
                    new_start,
                    new_end,
                    label
                ])

        chunks.append({
            "tokenized_text": chunk_tokens,
            "ner": chunk_ner
        })

        if token_end >= len(tokens):
            break

    return chunks
def chunk_data(data):
    def_data = []
    for entry in data:
        
        fragments = chunking_gliner(entry)
        def_data.append(fragments)
    return def_data
        
        
        

In [25]:
with open("./data/processed_train.json", "r") as file:
    data = json.load(file)
chunked = chunk_data(data)
with open("./data/processed_train_chunked.json", "w") as file:
    json.dump(chunked, file, indent=2)

### Pruebas

In [48]:

output = process_annotations(dataset)
bio = bionizar_datos(output)
chunked = chunk_data(bio)

In [31]:
bio_val = []

for entry in bio:

    *_, tokens, bio_tags, _ = entry.values()

    bio_val.append({
        "tokens": tokens,
        "bio": bio_tags
    })

In [33]:
with open("dev_chunked_bio_sp.json", "w")as file:
    json.dump(bio_val, file, indent=2)

In [49]:
c_0 = chunked[25][0]
c_tokens = c_0["chunk_tokens"]
c_s, c_e = c_0["chunk_start_tok"], c_0["chunk_end_tok"]
b_0 = bio[25]

b_tokens = b_0["bio"][c_s:c_e]
print(c_tokens)
print()
print(b_tokens)

['This', 'study', 'aimed', 'to', 'reveal', 'harm', 'of', 'exposure', 'to', 'indoor', 'air', 'pollution', 'to', 'cognitive', 'function', 'through', '"', 'gut-brain-axis', '"', 'among', 'rural', 'elderly', 'residents', '.', 'There', 'were', '120', 'participants', 'recruited', 'in', 'rural', 'villages', 'of', 'northwest', 'China', 'from', 'December', '2021', 'to', 'February', '2022', '.', 'The', 'cognitive', 'level', 'was', 'assessed', 'by', 'eight-item', 'ascertain', 'dementia', '(', 'AD', ')', 'questionnaire', ',', 'and', 'indoor', 'air', 'pollution', 'exposure', 'was', 'measured', 'by', 'air', 'quality', 'sensor', '.', 'Inflammatory', 'cytokines', 'and', 'oxidative', 'stress-related', 'index', 'were', 'detected', 'in', 'blood', 'serum', '.', 'Fecal', 'samples', 'were', 'collected', 'for', 'gut', 'microbiota', 'analysis', '.', 'The', '120', 'participants', 'were', 'divided', 'into', 'impaired', 'cognition', '(', 'AD8', ')', '(', '81', '/', '67', '.', '5', '%', ')', 'and', 'cognition', '

### Tokenizar dev dataset

In [1]:
import json
import re

In [17]:
with open("data/val_split.json", "r") as file:
    dev = json.load(file)

In [18]:
dev = process_annotations(dev)
dev = bionizar_datos(dev)
print(dev[0].keys())
dev = chunk_data(dev)
print(dev[1])

dict_keys(['pmid', 'location', 'tokens', 'bio', 'indexes'])


TypeError: 'dict' object is not callable

In [9]:
def split_sublists(data):
    output = []

    for sublist in data:
        for item in sublist:
            output.append(item)

    return output

data = split_sublists(dev)

In [10]:
print(data[0].keys())

dict_keys(['pmid', 'location', 'chunk_start_char', 'chunk_start_tok', 'chunk_end_char', 'chunk_end_tok', 'chunk_tokens', 'chunk_labels', 'chunk_spans'])


In [11]:
with open("test_chunked_reconstruct.json", "w")as file:
    json.dump(data, file, indent=2)

In [134]:
with open("predictions/18-05/mingxiaw21_base.json", "r") as file:
    preds = json.load(file)

In [135]:
print(len(preds[1]["tokens"]), len(preds[2]["tokens"]))
print(len(data[1]["chunk_tokens"]), len(data[2]["chunk_tokens"]))

256 62
256 62


In [125]:
from collections import defaultdict

def count_abstracts(data):
    counts = defaultdict(int)

    for item in data:
        if item.get("location") == "abstract":
            counts[item["pmid"]] += 1

    return counts


counts1 = count_abstracts(data)
counts2 = count_abstracts(preds)

common_pmids = set(counts1.keys()) & set(counts2.keys())

different_pmids = [
    pmid
    for pmid in common_pmids
    if counts1[pmid] != counts2[pmid]
]

print(len(different_pmids))

11


In [126]:
from collections import OrderedDict

def restructure_data(data):
    grouped = OrderedDict()

    for item in data:
        pmid = item["pmid"]
        location = item["location"]

        tokens = (
            item.get("tokens")
            or item.get("chunk_tokens")
            or item.get("tokenized_text")
            or []
        )

        if pmid not in grouped:
            grouped[pmid] = [pmid]

        grouped[pmid].append({
            f"tokenized_{location}": tokens
        })

    return list(grouped.values())

dev_chunked_partes = restructure_data(data)

In [132]:
dev_chunked_partes
with open("dev_chunked_partes.json", "w") as file:
    json.dump(dev_chunked_partes, file, indent = 2)

In [130]:
def restructure_to_tokens_bio(data):
    output = []

    for item in data:
        output.append({
            "tokens": item["chunk_tokens"],
            "bio": item["chunk_labels"]
        })

    return output

dev_chunked_bio = restructure_to_tokens_bio(data)

In [133]:
dev_chunked_bio
with open("dev_chunked_bio.json", "w") as file:
    json.dump(dev_chunked_bio, file, indent = 2)

In [153]:
print(data[0].keys())
print(preds[0].keys())

dict_keys(['pmid', 'location', 'chunk_start_char', 'chunk_start_tok', 'chunk_end_char', 'chunk_end_tok', 'chunk_tokens', 'chunk_labels', 'chunk_spans'])
dict_keys(['pmid', 'location', 'tokens', 'predictions'])


In [145]:
# for e1, e2 in zip(data, preds):
#     if e1["pmid"] != e2["pmid"] or e1["location"] != e2["location"]:
#         print(e2["pmid"])
#     else:
#         print(any(e1["chunk_tokens"] == e2["tokens"]) == False)

print(sum(1 for e1, e2 in zip(data, preds) if e1["chunk_labels"] == e2["predictions"]))

52


In [157]:
from collections import OrderedDict


def bio_to_spans(tokens, labels, spans):
    entities = []

    current_tokens = []
    current_spans = []
    current_label = None

    for token, bio_label, span in zip(tokens, labels, spans):

        if bio_label == "O":
            if current_label is not None:
                entities.append(build_entity(
                    current_tokens,
                    current_spans,
                    current_label
                ))

            current_tokens = []
            current_spans = []
            current_label = None
            continue

        prefix, label = bio_label.split("-", 1)

        if prefix == "B":
            if current_label is not None:
                entities.append(build_entity(
                    current_tokens,
                    current_spans,
                    current_label
                ))

            current_tokens = [token]
            current_spans = [span]
            current_label = label

        elif prefix == "I":

            if current_label == label:
                current_tokens.append(token)
                current_spans.append(span)

            else:
                if current_label is not None:
                    entities.append(build_entity(
                        current_tokens,
                        current_spans,
                        current_label
                    ))

                current_tokens = [token]
                current_spans = [span]
                current_label = label

    if current_label is not None:
        entities.append(build_entity(
            current_tokens,
            current_spans,
            current_label
        ))

    return entities


def build_entity(tokens, spans, label):

    start_idx = spans[0][0]
    end_idx = spans[-1][1] - 1

    return {
        "start_idx": start_idx,
        "end_idx": end_idx,
        "text_span": " ".join(tokens),
        "label": label
    }


def predictions_to_json(gold_data, pred_data):

    gold_index = {}

    for item in gold_data:

        key = (
            item["pmid"],
            item["location"],
            tuple(item["chunk_tokens"])
        )

        gold_index[key] = item["chunk_spans"]

    result = OrderedDict()

    for item in pred_data:

        pmid = item["pmid"]
        location = item["location"]
        tokens = item["tokens"]
        predictions = item["predictions"]

        key = (
            pmid,
            location,
            tuple(tokens)
        )

        if key not in gold_index:
            print(f"Missing spans for PMID {pmid}")
            continue

        spans = gold_index[key]

        entities = bio_to_spans(
            tokens,
            predictions,
            spans
        )

        if pmid not in result:
            result[pmid] = {
                "entities": []
            }

        for ent in entities:
            ent["location"] = location
            result[pmid]["entities"].append(ent)

    return result

In [158]:
output_json = predictions_to_json(data, preds)

In [159]:
with open("predictions/FORMATED.json", "w") as file:
    json.dump(output_json, file, indent = 2)

In [161]:
print(list(e1["chunk_tokens"] == e2["tokens"] for e1,e2 in zip(data, preds))) 

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, Tru

In [14]:
with open("test_chunked2.json", "w") as file:
    json.dump(resultado, file, indent = 2)

In [12]:
from collections import defaultdict

grouped = defaultdict(list)

for item in data:

    pmid = item["pmid"]

    # Copiar todo menos pmid
    chunk_data = {
        k: v
        for k, v in item.items()
        if k != "pmid"
    }

    grouped[pmid].append(chunk_data)

# Convertir al formato final
resultado = [
    {
        "pmid": pmid,
        "data": chunks
    }
    for pmid, chunks in grouped.items()
]